# Merge Diagnosis Summary

Reference document for the multi-view NF merging analysis on `exp_071_crop_shape`.
Consolidates §26+ of `multiview_posterior_tightening.ipynb` (rejecting earlier
hypotheses where superseded). All numbers are from the diagnostics in
`tinker/diag_*.py`.

**Setup**
* Model: `exp/exp_071_crop_shape` (BEDLAM-trained NF + extreme-cropping aug).
* Eval: 4D-Dress (OOD, subjects unseen) and BEDLAM val orbit-archviz-15-bbox44-smplx (IID).
* V = 4 views, S = 100 NF samples, N = 25 subjects per dataset (per diagnostic).


## 1. Headline empirical observations

### 1.1 Cross-view log-prob gap is small; IS weights partially collapse
Self-view NF log-prob is $\sim +25$ nats; cross-view (sample from view $i$, score under view $j \ne i$) is $\sim -5$ nats. **Gap $\sim 25$–$37$ nats.** ESS$/(V \cdot S) = 0.03$–$0.08$ (partial collapse, not uniform, not one-hot).

> **Rules out:** strict "thick slab" coverage failure, which would predict $10^2$–$10^3$ nat gaps and near-uniform ESS.

### 1.2 Flow is structurally overconfident
Calibrated against `checkpoints/sam-3d-body-dinov3/shape_scale_std.pt`:

| | OOD | IID |
|---|---:|---:|
| Median $\sigma_{\mathrm{pred}} / \sigma_{\mathrm{prior}}$ over 45 shape dims | $0.610$ | $0.659$ |
| Median $|z|$ over 45 shape dims | $1.62$ | $1.46$ |
| Calibrated reference | $\approx 1$ | $\approx 0.67$ |

Identical on IID and OOD ⇒ not a memorisation/leakage artefact; structural.

### 1.3 Uncertainty $\ne$ disagreement
The dims with the largest per-view $\sigma$ are *not* the dims that drive the cross-view penalty:

| Top-5 by $\sigma / \sigma_{\mathrm{prior}}$ | Top-5 by cross-view squared-residual |
|---|---|
| d43, d24, d7, d31, d28 | d10, d34, d25, d18, d11 |

Disagreement dims are **5–9× overconfident** (d25 $|z| = 6.0$, d34 $|z| = 5.5$, d10 $|z| = 4.0$). A subset of uncertainty dims (d31, d28) is properly calibrated. The fusion failure is co-located with the calibration failure.

### 1.4 Top-ranked dims are all shape PCs; geometrically near-null
All 10 top dims (uncertainty and disagreement) are shape PCs ($< 45$). At $\pm 3 \sigma_{\mathrm{prior}}$ they produce only **1–15 mm peak vertex displacement** on the neutral mesh. The flow is most confident on the dims that carry the least image evidence.

### 1.5 Regressor mean has prior-pull bias on OOD
Subject heights (mm), neutral-pose mesh:

| | OOD | IID |
|---|---:|---:|
| GT pop. $\sigma$ | $46$ | $34$ |
| $\mu$-pred pop. $\sigma$ | $\mathbf{16}$ | $21$ |
| Per-subject bias $\langle \mu_j - \mathrm{GT} \rangle_j$ | $\mathbf{-84}$ | $-0.5$ |
| $\mu$ across-V $\sigma$ | $13$ | $30$ |
| Sample within-V $\sigma$ | $29$ | $32$ |
| Ratio (across-V / within-V) | $0.45$ | $0.94$ |

On OOD, every subject's height collapses onto a $\sim 1677$ mm prior. Per-view variance is appropriate; the **regressor mean** is what's biased, not the flow conditional. On IID, views genuinely disagree on height by $\sim 30$ mm.

### 1.6 Across-view disagreement on scale dims is rank $\approx 2$
PCA on $(\mu_j - \langle\mu\rangle_j)$ over the 10 flow-modulated scale dims:

| Dataset | PC1 | PC1+2 | PC1 sign concordance | uniform-axis var fraction |
|---|---:|---:|---:|---:|
| OOD | $57\%$ | $79\%$ | $+0.60$ | $17.7\%$ |
| IID | $69\%$ | $83\%$ | $\mathbf{0.00}$ | $\mathbf{10.0\%}$ |

The disagreement subspace is 2-D, not 10-D. Sign concordance $0.00$ on IID (s3 $+0.46$ vs s11 $-0.48$) means PC1 is a *rotation* of body proportions, not pure overall-magnitude scaling. The flow's sample-mean PC1 is closer to a uniform-direction (concordance $+1.00$ on OOD), so $\bar\beta_j$ already projects per-view answers onto a more "physical" overall-scale direction than $\mu_j$.


## 2. Diagnosis — three coexisting failure modes

The merge fails for different reasons on different dims; no single-mode story explains everything.

### A. Posterior overconfidence on disagreement shape dims (d10, d18, d25, d34, d11)
Per-view modes confidently disagree. The flow's $\sigma$ around each mode is $5$–$9\times$ too narrow to admit the other view's posterior, so cross-view IS weights collapse onto in-mode samples. The bias lives in the flow's **narrow conditional**, not in the regressor mean. Replacing $\mu_j$ with $\bar\beta_j$ does not help because $\bar\beta_j$ is also centred at the over-confident mode.

### B. Regressor prior-pull on globally-supervised dims (height / scale)
On OOD, the regressor squeezes the per-view mean onto the BEDLAM-prior body shape, losing $\sim 80$ mm of GT signal. Per-view $\sigma$ on height is appropriate (ratio $\le 1$); flow samples cover the cross-view spread. The bias lives in the **regressor mean**, not the flow. Bias-corrected oracle ($\bar\beta_j$) recovers $\sim 40$ mm on tall subjects but not all.

### C. Correlated-bias-within-low-rank-subspace on scale block
Cross-view disagreement on the 10 scale dims is $\approx 2$-D, but every view is biased *together* within this subspace toward the prior body shape (failure mode B applied to a 2-D subspace). Cross-view averaging confirms the bias rather than cancelling it. The subspace dimension is not the bottleneck — view correlation within it is.

### Ruled out by data
* Strict IS coverage failure (§16 of source doc).
* Subject-identity leakage / memorisation.
* Distribution-shift sensitivity as the dominant driver.
* Pure-magnitude scale ambiguity (rejected: rotation in scale space, not pure scaling).


## 3. Proposed solutions

### Cheap (no retraining)

1. **Per-dim post-hoc temperature rescaling.** Estimate $\hat\tau_d$ on a held-out IID set such that $\hat\tau_d \cdot \sigma_{\mathrm{pred},d}$ matches the empirical $|z|$. Apply $\hat\tau_d \in \{2, 5, 9\}\times$ to disagreement dims and $\approx 1\times$ to calibrated dims (height, d28, d31). Use rescaled $\sigma$ inside the cross-view IS / Langevin computation. **Targets A.**

2. **Bias-corrected oracle ($\bar\beta_j$) instead of $\mu_j$, on dims where it helps.** Empirically helps height ($\sim 40$ mm on tall OOD subjects). Does not help on disagreement dims. Apply per-dim, not globally.

### Diagnostic / partition-test runs (no retraining)

3. **`gaussian` and `langevin` merge methods.** Expect both to inherit failure modes A and B (Gaussian over-trusts each view; Langevin converges to an over-narrow joint mode). Useful as control to confirm calibration is the bottleneck rather than the merge algorithm.

4. **Per-dim cross-view-weight decomposition.** Compute the IS weight contribution per dim and per view-pair to confirm A is driven by disagreement-shape dims and the height/scale failure is driven by mode B.

### Retraining

5. **Calibration regulariser on the flow.** Per-dim penalty pushing $\sigma_d$ to match empirical $|\beta_{gt,d} - \mu_d|$. Sharper variant: KL term against a broad prior on dims with small rendering Jacobian. **Targets A.**

6. **Bias-aware regressor head.** Either output per-dim "uncertainty about my own mean", or use a residual-prediction architecture so the OOD prior pull cannot squeeze every prediction onto the training mean. **Targets B.**

### De-prioritised
* Lowering flow $D$ / dropping scale dims (the scale dims aren't where the merge fails as 2-D fusion has signal in principle; the failure on scale is correlated bias, not over-modelling).
* Joint-Gaussian-proposal IS (depends on the §16 coverage-failure premise that does not hold).
* Replacing $\mu_j$ with $\bar\beta_j$ uniformly across all dims (helps height, hurts dims where flow mode is biased).


## 4. Open questions

1. **How much of the merge gap closes if both A and B are addressed?** Step 1 + step 2 (cheap) gives an upper bound for the no-retraining gain.
2. **Is the calibration failure (mode A) intrinsic to NLL training, or curable by adding an information-aware regulariser without changing the loss form?** Step 5 isolates this.
3. **Does the regressor prior-pull (mode B) also affect shape dims that we've classified as overconfident, or is it specific to scale?** Per-dim bias measurement on the disagreement shape dims would tell us whether $\mu_d$ is biased toward zero on OOD as well.
4. **What is the rank of disagreement on the 45 shape dims?** §40 covered scale only; the equivalent PCA on shape would tell us whether the disagreement-shape dims (d10, d18, d25, d34) align in some low-D subspace or are independent.


## 5. Diagnostic scripts

| Script | What it measures |
|---|---|
| `tinker/diag_cross_view_logp.py` | Self vs cross-view NF log-prob gap, ESS of tempered IS weights |
| `tinker/diag_dim_calibration.py` | Per-dim $\sigma / \sigma_{\mathrm{prior}}$ |
| `tinker/diag_calibration_iid_ood.py` | $|z|$-score per dim, IID vs OOD |
| `tinker/diag_effective_dim.py` | Top dims by uncertainty and disagreement |
| `tinker/viz_top_dims.py` | Body-shape effect of top dims; vertex-displacement heatmaps |
| `tinker/diag_height_per_view.py` | Per-view subject-height predictions |
| `tinker/diag_scale_rank.py` | Rank of across-view scale disagreement |

All scripts share the same loader pattern (`exp/exp_071_crop_shape`, `last.ckpt`, calibrated stds from `checkpoints/sam-3d-body-dinov3/shape_scale_std.pt`).
